Below is the **existing Hybrid RAG code**, expanded to show the main reranking approaches and **when to use each one**.

```python
# Install:
# pip install langchain langchain-community langchain-openai
# pip install rank-bm25 faiss-cpu sentence-transformers

from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain.retrievers import EnsembleRetriever

# --------------------------------------------------
# DOCUMENTS
# --------------------------------------------------

documents = [
    "Python is widely used for machine learning and data science.",
    "You can reset your password from the account settings page.",
    "Machine learning algorithms learn patterns from training data.",
    "To reset your password, click the forgot password option.",
    "Python libraries include NumPy, Pandas and Scikit-learn.",
    "Contact the administrator if your password reset does not work."
]

query = "How can I reset my password?"


# ==================================================
# 1. BM25 RETRIEVER
# ==================================================

bm25 = BM25Retriever.from_texts(documents)
bm25.k = 5


# ==================================================
# 2. SEMANTIC RETRIEVER
# ==================================================

embeddings = OpenAIEmbeddings()

vector_db = FAISS.from_texts(
    documents,
    embeddings
)

semantic = vector_db.as_retriever(
    search_kwargs={"k": 5}
)


# ==================================================
# 3. HYBRID RETRIEVAL
# ==================================================

hybrid = EnsembleRetriever(
    retrievers=[bm25, semantic],
    weights=[0.5, 0.5]
)

retrieved_docs = hybrid.invoke(query)

print("\n--- HYBRID RESULTS ---")

for doc in retrieved_docs:
    print(doc.page_content)
```

Now you can apply different reranking techniques to these retrieved documents.

---

# 1. Cross-Encoder Reranking

A **cross-encoder** directly looks at:

```text
Query + Document
       ↓
Relevance Score
```

### Code

```python
from sentence_transformers import CrossEncoder

model = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

pairs = [
    (query, doc.page_content)
    for doc in retrieved_docs
]

scores = model.predict(pairs)

reranked = sorted(
    zip(retrieved_docs, scores),
    key=lambda x: x[1],
    reverse=True
)

print("\n--- CROSS-ENCODER RERANKING ---")

for doc, score in reranked[:3]:
    print(score, "->", doc.page_content)
```

### When to use?

Use **Cross-Encoder** when:

* Retrieval quality is important.
* You have a manageable number of retrieved documents.
* You want better relevance than basic embedding similarity.
* You can afford additional computation.

**Very common choice for production RAG.**

---

# 2. BGE Reranker

BGE provides open-source reranking models.

Example model:

```text
BAAI/bge-reranker-v2-m3
```

### Code

```python
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-v2-m3"
)

pairs = [
    (query, doc.page_content)
    for doc in retrieved_docs
]

scores = reranker.predict(pairs)

reranked = sorted(
    zip(retrieved_docs, scores),
    key=lambda x: x[1],
    reverse=True
)

for doc, score in reranked[:3]:
    print(score, "->", doc.page_content)
```

### When to use?

Use **BGE Reranker** when:

* You want an **open-source** solution.
* You don't want to depend on a commercial API.
* You can run the model locally.
* You need strong semantic relevance.

---

# 3. Cohere Reranker

Instead of running the reranker locally, you can use a reranking API.

```python
from langchain_cohere import CohereRerank
from langchain.retrievers import ContextualCompressionRetriever

reranker = CohereRerank(
    model="rerank-v3.5",
    top_n=3
)

reranked_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=hybrid
)

results = reranked_retriever.invoke(query)

for doc in results:
    print(doc.page_content)
```

### When to use?

Use a **Cohere Reranker** when:

* You want a managed API.
* You don't want to host the reranking model.
* You want simple integration with LangChain.
* Latency and API cost are acceptable.

---

# 4. LLM-Based Reranking

Here, an LLM evaluates each document.

Concept:

```text
Query + Document
       ↓
      LLM
       ↓
Relevance Score
       ↓
Ranking
```

Example:

```python
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

for doc in retrieved_docs:

    prompt = f"""
    Query: {query}

    Document:
    {doc.page_content}

    Give a relevance score from 0 to 10.
    Return only the number.
    """

    score = llm.invoke(prompt)

    print(doc.page_content)
    print("Score:", score.content)
```

### When to use?

Use **LLM reranking** when:

* Relevance requires complex reasoning.
* Documents need contextual understanding.
* You have a small number of candidates.
* Quality is more important than latency/cost.

**Disadvantage:** More expensive and slower than a dedicated reranker.

---

# 5. Reciprocal Rank Fusion — RRF

RRF is particularly useful for combining:

```text
BM25 ranking
     +
Semantic ranking
     ↓
    RRF
     ↓
Combined ranking
```

Formula:

```text
RRF Score = Σ 1 / (k + rank)
```

Simple implementation:

```python
def rrf(rankings, k=60):

    scores = {}

    for ranking in rankings:

        for rank, doc in enumerate(ranking, start=1):

            text = doc.page_content

            scores[text] = scores.get(text, 0) + \
                           1 / (k + rank)

    return sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )
```

### When to use?

Use **RRF** when:

* You have multiple retrievers.
* You want to combine BM25 + semantic results.
* The individual retrievers produce different rankings.
* You don't want to worry about different score scales.

**Very useful for Hybrid RAG.**

---

# 6. Weighted Score Fusion

Here you explicitly give weights.

For example:

```text
Final Score =
0.7 × Semantic Score
+
0.3 × BM25 Score
```

Example:

```python
semantic_weight = 0.7
bm25_weight = 0.3

final_score = (
    semantic_weight * semantic_score
    +
    bm25_weight * bm25_score
)
```

### When to use?

Use **Weighted Fusion** when:

* You know one retriever is more reliable.
* Your domain depends heavily on exact keywords.
* You want control over retrieval behavior.

Example:

**Legal documents:**

```text
BM25 = 0.6
Semantic = 0.4
```

because exact legal terms can be important.

---

# 7. MMR — Maximum Marginal Relevance

MMR tries to select documents that are:

```text
Relevant
     +
Diverse
```

Instead of returning:

```text
Document 1 → Password reset
Document 2 → Password reset
Document 3 → Password reset
```

MMR tries to give:

```text
Document 1 → Password reset
Document 2 → Account recovery
Document 3 → Contact administrator
```

Example:

```python
semantic_mmr = vector_db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 10,
        "lambda_mult": 0.5
    }
)

results = semantic_mmr.invoke(query)

for doc in results:
    print(doc.page_content)
```

### When to use?

Use **MMR** when:

* Retrieved chunks contain many duplicates.
* You want diverse information.
* Your documents contain repetitive content.
* You want broader context for the LLM.

---

# Which one should you use?

| Reranking           | Best use case                                     |
| ------------------- | ------------------------------------------------- |
| **Cross-Encoder**   | Best general-purpose reranking                    |
| **BGE Reranker**    | Open-source/local RAG                             |
| **Cohere Reranker** | Easy managed/API solution                         |
| **LLM Reranking**   | Complex reasoning/relevance                       |
| **RRF**             | Combining BM25 + semantic rankings                |
| **Weighted Fusion** | When you want manual control                      |
| **MMR**             | Removing duplicate results / increasing diversity |

### Recommended RAG architecture

For a strong general-purpose RAG system:

```text
                    User Query
                        ↓
             ┌──────────┴──────────┐
             ↓                     ↓
           BM25              Semantic Search
             ↓                     ↓
             └──────────┬──────────┘
                        ↓
                 Hybrid Retrieval
                        ↓
                       RRF
                        ↓
                Top 10-20 chunks
                        ↓
                  Cross-Encoder
                    Reranker
                        ↓
                  Top 3-5 chunks
                        ↓
                       LLM
                        ↓
                    Final Answer
```

**For learning order, I recommend:**
**BM25 → Semantic Search → Hybrid → RRF → Cross-Encoder → BGE/Cohere → LLM Reranking → MMR.**
